In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
from mlflow import MlflowClient


client = MlflowClient(tracking_uri="http://127.0.0.1:3000")

class MyTrainDatset(Dataset):
    def __init__(self, num_of_samples: int = 20048):
        self.size = num_of_samples
        self.data = [(torch.rand(size=(20,)), torch.randint(0,2, size=(1,)).squeeze(0)  )for i in range(num_of_samples)]
        
    def __len__(self):
        return self.size
    
    def __getitem__(self, index):
        return self.data[index]
        
        
def train_objs():
    dataset = MyTrainDatset()
    model = nn.Linear(in_features=20, out_features=2)

    optimizer = torch.optim.Adam(params=model.parameters(), lr=1e-3)
    return model, dataset, optimizer

def prepare_dataloader(dataset:Dataset, batch_size:int):
    return DataLoader(dataset=dataset, batch_size=batch_size, shuffle=True, pin_memory=True)

class Trainer():
    def __init__(self, model, dataset:DataLoader, optimizer, gpu_id:int, save_every:int):
        self.gpu_id = gpu_id
        self.model = model.to(gpu_id)
        self.optimizer = optimizer
        self.train_data = dataset
        self.save_every = save_every
    
    def train(self, max_epochs:int):
        for epoch in range(max_epochs):
            print(f"GPU : {self.gpu_id} | Epoch : {epoch} | ")
            for features, target in self.train_data:
                features = features.to(self.gpu_id)
                target = target.to(self.gpu_id)
                logits = self.model(features)
                loss = F.cross_entropy(logits, target)
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()
    
device = 0 # Not a string
#device = "mps:0" # Not a string
batch_size = 32
save_every = 2
model, dataset, optimizer = train_objs()
train_data = prepare_dataloader(dataset=dataset, batch_size=batch_size)
trainer = Trainer(model,train_data, optimizer, device, save_every )

trainer.train(max_epochs=5)


GPU : 0 | Epoch : 0 | 


/Users/sachinmurali/anaconda3/envs/tensorflows/lib/python3.11/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


GPU : 0 | Epoch : 1 | 
GPU : 0 | Epoch : 2 | 
GPU : 0 | Epoch : 3 | 
GPU : 0 | Epoch : 4 | 


In [36]:
from mlflow import MlflowClient


client = MlflowClient(tracking_uri="http://127.0.0.1:3000")


client

In [37]:
client.search_experiments()

[]

In [38]:
# Provide an Experiment description that will appear in the UI
experiment_description = (
    "This is the grocery forecasting project. "
    "This experiment contains the produce models for apples."
)

# Provide searchable tags that define characteristics of the Runs that
# will be in this Experiment
experiment_tags = {
    "project_name": "grocery-forecasting",
    "store_dept": "produce",
    "team": "stores-ml",
    "project_quarter": "Q3-2023",
    "mlflow.note.content": experiment_description,
}

# Create the Experiment, providing a unique name
produce_apples_experiment = client.create_experiment(
    name="Apple_Modelss", tags=experiment_tags
)

In [39]:
client.search_experiments(filter_string="tags.`project_name` = 'grocery-forecasting'")

[<Experiment: artifact_location='mlflow-artifacts:/385143182107718454', creation_time=1752639460331, experiment_id='385143182107718454', last_update_time=1752639460331, lifecycle_stage='active', name='Apple_Modelss', tags={'mlflow.note.content': 'This is the grocery forecasting project. This '
                         'experiment contains the produce models for apples.',
  'project_name': 'grocery-forecasting',
  'project_quarter': 'Q3-2023',
  'store_dept': 'produce',
  'team': 'stores-ml'}>]

In [40]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta


def generate_apple_sales_data_with_promo_adjustment(
    base_demand: int = 1000, n_rows: int = 5000
):
    """
    Generates a synthetic dataset for predicting apple sales demand with seasonality
    and inflation.

    This function creates a pandas DataFrame with features relevant to apple sales.
    The features include date, average_temperature, rainfall, weekend flag, holiday flag,
    promotional flag, price_per_kg, and the previous day's demand. The target variable,
    'demand', is generated based on a combination of these features with some added noise.

    Args:
        base_demand (int, optional): Base demand for apples. Defaults to 1000.
        n_rows (int, optional): Number of rows (days) of data to generate. Defaults to 5000.

    Returns:
        pd.DataFrame: DataFrame with features and target variable for apple sales prediction.

    Example:
        >>> df = generate_apple_sales_data_with_seasonality(base_demand=1200, n_rows=6000)
        >>> df.head()
    """

    # Set seed for reproducibility
    np.random.seed(9999)

    # Create date range
    dates = [datetime.now() - timedelta(days=i) for i in range(n_rows)]
    dates.reverse()

    # Generate features
    df = pd.DataFrame(
        {
            "date": dates,
            "average_temperature": np.random.uniform(10, 35, n_rows),
            "rainfall": np.random.exponential(5, n_rows),
            "weekend": [(date.weekday() >= 5) * 1 for date in dates],
            "holiday": np.random.choice([0, 1], n_rows, p=[0.97, 0.03]),
            "price_per_kg": np.random.uniform(0.5, 3, n_rows),
            "month": [date.month for date in dates],
        }
    )

    # Introduce inflation over time (years)
    df["inflation_multiplier"] = (
        1 + (df["date"].dt.year - df["date"].dt.year.min()) * 0.03
    )

    # Incorporate seasonality due to apple harvests
    df["harvest_effect"] = np.sin(2 * np.pi * (df["month"] - 3) / 12) + np.sin(
        2 * np.pi * (df["month"] - 9) / 12
    )

    # Modify the price_per_kg based on harvest effect
    df["price_per_kg"] = df["price_per_kg"] - df["harvest_effect"] * 0.5

    # Adjust promo periods to coincide with periods lagging peak harvest by 1 month
    peak_months = [4, 10]  # months following the peak availability
    df["promo"] = np.where(
        df["month"].isin(peak_months),
        1,
        np.random.choice([0, 1], n_rows, p=[0.85, 0.15]),
    )

    # Generate target variable based on features
    base_price_effect = -df["price_per_kg"] * 50
    seasonality_effect = df["harvest_effect"] * 50
    promo_effect = df["promo"] * 200

    df["demand"] = (
        base_demand
        + base_price_effect
        + seasonality_effect
        + promo_effect
        + df["weekend"] * 300
        + np.random.normal(0, 50, n_rows)
    ) * df[
        "inflation_multiplier"
    ]  # adding random noise

    # Add previous day's demand
    df["previous_days_demand"] = df["demand"].shift(1)
    df["previous_days_demand"].fillna(
        method="bfill", inplace=True
    )  # fill the first row

    # Drop temporary columns
    df.drop(columns=["inflation_multiplier", "harvest_effect", "month"], inplace=True)

    return df

In [41]:
data = generate_apple_sales_data_with_promo_adjustment(base_demand=1_000, n_rows=1_000)

data[-20:]

/var/folders/c7/z9xcy1l1481bm3hgrydzvlxm0000gn/T/ipykernel_3627/3859398416.py:89: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["previous_days_demand"].fillna(
/var/folders/c7/z9xcy1l1481bm3hgrydzvlxm0000gn/T/ipykernel_3627/3859398416.py:89: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["previous_days_demand"].fillna(


,date,average_temperature,rainfall,weekend,holiday,price_per_kg,promo,demand,previous_days_demand
980,2025-06-27 09:47:40.387901,34.130183,1.454065,0,0,1.449177,0,999.306290,1029.418398
981,2025-06-28 09:47:40.387901,32.353643,9.462859,1,0,2.856503,0,1169.129427,999.306290
982,2025-06-29 09:47:40.387900,18.816833,0.391470,1,0,1.326429,0,1317.616709,1169.129427
983,2025-06-30 09:47:40.387900,34.533012,2.120477,0,0,0.970131,0,1068.802075,1317.616709
984,2025-07-01 09:47:40.387899,23.057202,2.365705,0,0,1.049931,0,1019.486305,1068.802075
985,2025-07-02 09:47:40.387899,34.810165,3.089005,0,0,2.035149,0,1002.564672,1019.486305
986,2025-07-03 09:47:40.387898,29.208905,3.673292,0,0,2.518098,0,1086.143402,1002.564672
987,2025-07-04 09:47:40.387898,16.428676,4.077782,0,0,1.268979,0,1093.207186,1086.143402
988,2025-07-05 09:47:40.387897,32.067512,2.734454,1,0,0.762317,0,1396.939894,1093.207186
989,2025-07-06 09:47:40.387897,31.938203,13.883486,1,0,1.153301,0,1321.409540,1396.939894


In [42]:
import mlflow

# Sets the current active experiment to the "Apple_Models" experiment and
# returns the Experiment metadata
apple_experiment = mlflow.set_experiment("Apple_Model")

# Define a run name for this iteration of training.
# If this is not set, a unique name will be auto-generated for your run.
run_name = "apples_rf_test"

# Define an artifact path that the model will be saved to.
artifact_path = "rf_apples"

In [43]:
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# Split the data into features and target and drop irrelevant date field and target field
X = data.drop(columns=["date", "demand"])
y = data["demand"]

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

params = {
    "n_estimators": 100,
    "max_depth": 6,
    "min_samples_split": 10,
    "min_samples_leaf": 4,
    "bootstrap": True,
    "oob_score": False,
    "random_state": 888,
}

# Train the RandomForestRegressor
rf = RandomForestRegressor(**params)

# Fit the model on the training data
rf.fit(X_train, y_train)

# Predict on the validation set
y_pred = rf.predict(X_val)

# Calculate error metrics
mae = mean_absolute_error(y_val, y_pred)
mse = mean_squared_error(y_val, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_val, y_pred)

# Assemble the metrics we're going to write into a collection
metrics = {"mae": mae, "mse": mse, "rmse": rmse, "r2": r2}

# Initiate the MLflow run context
with mlflow.start_run(run_name=run_name) as run:
    # Log the parameters used for the model fit
    mlflow.log_params(params)

    # Log the error metrics that were calculated during validation
    mlflow.log_metrics(metrics)

    # Log an instance of the trained model for later use
    mlflow.sklearn.log_model(sk_model=rf, input_example=X_val, name=artifact_path)

/Users/sachinmurali/anaconda3/envs/tensorflows/lib/python3.11/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
